# 🧪 실험 3: Transfer Learning + 증강 + Class Weight

---

## 이 모델이 하는 일

실험 1, 2와 동일하게 **반려동물 피부 사진 → 7가지 질환 분류**를 수행합니다.

```
📸 반려동물 피부 사진 입력
        ↓
    [EfficientNetB0 + 커스텀 분류층 + 종별 가중치]
        ↓
    A4 농포/여드름 : 92.4%  ← 예측 결과 (반려묘도 잘 맞춤!)
```

## 실험 1, 2와 무엇이 다른가?

| 항목 | 실험 1 | 실험 2 | 실험 3 (이번) |
|------|--------|--------|-------------|
| **모델** | 직접 설계 CNN | EfficientNetB0 | EfficientNetB0 |
| **증강** | ❌ 없음 | ✅ 적용 | ✅ 적용 |
| **Class Weight** | ❌ 없음 | ❌ 없음 | ✅ **적용** |
| **핵심 질문** | 분류 가능한가? | 성능이 올라가는가? | **소수 클래스(반려묘) 성능이 올라가는가?** |

## 왜 Class Weight가 필요한가?

현재 데이터에 **종별 불균형**이 존재합니다:

```
반려견(D): 31,010장 (88.6%) ← 압도적으로 많음
반려묘(C):  3,990장 (11.4%) ← 매우 적음
```

Class Weight 없이 학습하면 모델이 **반려견 데이터에 편향**되어, 반려묘 이미지를 잘 분류하지 못할 수 있습니다.

## Class Weight 동작 원리

```
반려견(D) 가중치: 0.563  → 흔한 데이터니까 loss 기여도를 줄임
반려묘(C) 가중치: 4.443  → 희귀한 데이터니까 loss 기여도를 4.4배 높임
```

모델이 반려묘 이미지를 틀리면 **4.4배 큰 벌점(loss)**을 받으므로, 반려묘도 잘 맞추도록 학습됩니다.

## 기대 효과

- 전체 정확도는 실험 2와 비슷하거나 약간 변동
- **반려묘 클래스의 Recall/F1이 향상** (소수 클래스 성능 개선)
- 더 공정한 모델 → **실제 서비스 배포에 적합한 최종 후보**

---


## 0. 라이브러리 임포트 및 설정

실험 2와 동일합니다. 추가로 **class_weight 계산을 위한 sklearn** 유틸리티를 임포트합니다.


In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras

from keras.applications import EfficientNetB0
from keras.models import Model
from keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    Input
)
from keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight  # ⭐ 실험 3 추가: 클래스 가중치 자동 계산
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print(f"TensorFlow 버전: {tf.__version__}")
print(f"GPU 사용 가능: {len(tf.config.list_physical_devices('GPU')) > 0}")


TensorFlow 버전: 2.21.0
GPU 사용 가능: False


## 1. 경로 설정 및 데이터 로딩

실험 1, 2와 **동일한 데이터, 동일한 분할**을 사용합니다.  
실험 간 공정한 비교를 위해 데이터는 절대 바꾸지 않습니다.


In [8]:
ROOT      = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
FIG_DIR   = ROOT / 'outputs' / 'figures'
MODEL_DIR = ROOT / 'models'

FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# CSV 로딩
df = pd.read_csv(PROCESSED / 'dataset_cleaned.csv')
target_col = 'img_path'

df_train = df[df['split'] == 'train'].copy()
df_val   = df[df['split'] == 'val'].copy()
df_test  = df[df['split'] == 'test'].copy()

print(f"✅ 데이터 로드 완료")
print(f"  train : {len(df_train):,}장")
print(f"  val   : {len(df_val):,}장")
print(f"  test  : {len(df_test):,}장")


✅ 데이터 로드 완료
  train : 24,492장
  val   : 5,249장
  test  : 5,246장


## 2. 하이퍼파라미터 설정

실험 2와 동일합니다. Class Weight는 별도 단계에서 계산합니다.


In [9]:
IMG_SIZE      = (224, 224)   # EfficientNetB0 ImageNet 사전학습 크기와 동일 (exp2와 통일)
BATCH_SIZE    = 64
EPOCHS        = 20
NUM_CLASSES   = 7
LEARNING_RATE = 0.0001       # 실험 2와 동일 (사전학습 모델 미세조정용)

print(f"이미지 크기  : {IMG_SIZE}")
print(f"배치 크기    : {BATCH_SIZE}")
print(f"최대 에폭    : {EPOCHS}")
print(f"클래스 수    : {NUM_CLASSES}")
print(f"학습률       : {LEARNING_RATE}")

이미지 크기  : (224, 224)
배치 크기    : 64
최대 에폭    : 20
클래스 수    : 7
학습률       : 0.0001


## 3. Class Weight 계산 ⭐ (실험 3 핵심)

이 단계가 **실험 2와의 유일한 차이점**입니다.

### 실험 3에서 적용하는 가중치

| 가중치 | 기준 | 적용 방식 | 목적 |
|--------|------|----------|------|
| **class_weight** | 질환별 (A1~A7) | `model.fit(class_weight=...)` | 질환 클래스 불균형 보정 ✅ |
| species_weight | 종별 (반려견/반려묘) | 참고용 (학습 미적용) | 불균형 규모 파악 |

> 💡 종별 불균형(`species_weight`)은 데이터 현황 파악을 위해 계산하되, 학습에는 적용하지 않습니다.  
> 종별 불균형까지 보정하려면 `sample_weight`로 결합 가중치를 적용할 수 있으나, 실험 3에서는 질환별 `class_weight`만 사용합니다.

### sample_weight vs class_weight 차이

- `class_weight`: **클래스 단위** 가중치. A1이 적으면 A1 전체에 높은 가중치
- `sample_weight`: **샘플(이미지) 단위** 가중치. 같은 A1이라도 반려묘면 가중치가 더 높음

In [10]:
# ── 3-1. 종별(species) 가중치 계산 (참고용, 학습 미적용) ──
# 반려견(D): 31,010장 → 가중치 낮게 / 반려묘(C): 3,990장 → 가중치 높게
# 불균형 규모 파악용으로만 사용

species_classes = np.array(df_train['species'].unique())
species_weights = compute_class_weight(
    class_weight='balanced',
    classes=species_classes,
    y=df_train['species']
)
species_weight_dict = dict(zip(species_classes, species_weights))

print("=== 종별(species) 가중치 (참고용) ===")
for k, v in species_weight_dict.items():
    label = '반려묘' if k == 'C' else '반려견'
    count = len(df_train[df_train['species'] == k])
    print(f"  {k} ({label}): {v:.3f}  (데이터 {count:,}장)")

=== 종별(species) 가중치 (참고용) ===
  D (반려견): 0.563  (데이터 21,736장)
  C (반려묘): 4.443  (데이터 2,756장)


In [11]:
# ── 3-2. 질환별(lesion) 클래스 가중치 계산 ──
# Generator의 class_indices 순서에 의존하지 않도록 이름→가중치 맵으로 저장
# class_weight_dict는 Generator 생성 후 class_indices를 참조하여 확정 (Section 4)

lesion_classes = sorted(df_train['lesion'].unique())
lesion_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(lesion_classes),
    y=df_train['lesion']
)
lesion_weight_map = dict(zip(lesion_classes, lesion_weights))  # {클래스명: 가중치}

print("=== 질환별(lesion) 클래스 가중치 ===")
for cls, w in lesion_weight_map.items():
    count = len(df_train[df_train['lesion'] == cls])
    print(f"  {cls} → 가중치 {w:.3f}  (데이터 {count:,}장)")

=== 질환별(lesion) 클래스 가중치 ===
  A1 → 가중치 1.000  (데이터 3,499장)
  A2 → 가중치 1.000  (데이터 3,499장)
  A3 → 가중치 1.000  (데이터 3,498장)
  A4 → 가중치 1.001  (데이터 3,497장)
  A5 → 가중치 1.000  (데이터 3,500장)
  A6 → 가중치 1.000  (데이터 3,499장)
  A7 → 가중치 1.000  (데이터 3,500장)


## 4. 데이터 Generator 생성 (증강 적용)

실험 2와 **동일한 증강 설정**입니다.  
Class Weight는 Generator가 아니라 **model.fit()에서 적용**합니다.


In [12]:
# Train용: 정규화 + 증강 (실험 2와 동일)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=30,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)

# Val/Test용: 정규화만
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("✅ ImageDataGenerator 정의 완료 (실험 2와 동일)")


✅ ImageDataGenerator 정의 완료 (실험 2와 동일)


In [13]:
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    x_col=target_col,
    y_col='lesion',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val,
    x_col=target_col,
    y_col='lesion',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col=target_col,
    y_col='lesion',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n✅ Generator 생성 완료")
print(f"클래스 인덱스: {train_generator.class_indices}")

# Generator의 class_indices를 참조하여 class_weight_dict 확정
# sorted() 순서에 의존하지 않고 Generator가 실제 사용하는 인덱스와 정확히 매핑
class_weight_dict = {
    idx: lesion_weight_map[cls]
    for cls, idx in train_generator.class_indices.items()
}

print(f"\n✅ class_weight_dict 확정")
for idx, w in sorted(class_weight_dict.items()):
    cls = [c for c, i in train_generator.class_indices.items() if i == idx][0]
    print(f"  인덱스 {idx} ({cls}): {w:.3f}")

Found 24492 validated image filenames belonging to 7 classes.
Found 5249 validated image filenames belonging to 7 classes.
Found 5246 validated image filenames belonging to 7 classes.

✅ Generator 생성 완료
클래스 인덱스: {'A1': 0, 'A2': 1, 'A3': 2, 'A4': 3, 'A5': 4, 'A6': 5, 'A7': 6}

✅ class_weight_dict 확정
  인덱스 0 (A1): 1.000
  인덱스 1 (A2): 1.000
  인덱스 2 (A3): 1.000
  인덱스 3 (A4): 1.001
  인덱스 4 (A5): 1.000
  인덱스 5 (A6): 1.000
  인덱스 6 (A7): 1.000


## 5. EfficientNetB0 + 커스텀 분류층 정의

실험 2와 **완전히 동일한 모델 구조**입니다.  
차이점은 모델 구조가 아니라 **학습 시 class_weight 적용 여부**입니다.

```
입력 (224×224×3)
    ↓
EfficientNetB0 (ImageNet, freeze)
    ↓
GlobalAveragePooling2D → 1280
    ↓
Dense(256, relu) → Dropout(0.3)
    ↓
Dense(7, softmax) → [A1~A7 확률]
```

In [14]:
# EfficientNetB0 사전학습 모델 로드
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)   # 사전학습과 동일한 입력 크기 (exp2와 통일)
)

# 사전학습 가중치 고정
base_model.trainable = False

# 커스텀 분류층 연결
inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

print(f"✅ 모델 구성 완료 (실험 2와 동일 구조)")
print(f"  총 레이어 수: {len(model.layers)}")

✅ 모델 구성 완료 (실험 2와 동일 구조)
  총 레이어 수: 6


## 6. 모델 컴파일

실험 2와 동일합니다. Class Weight는 compile이 아니라 **fit()에서 적용**합니다.


In [15]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 7. 콜백(Callbacks) 설정

저장 경로만 **`exp3_weighted.h5`**로 변경합니다.


In [16]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=str(MODEL_DIR / 'exp3_weighted.h5'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print("✅ 콜백 설정 완료")
print(f"  모델 저장 경로: {MODEL_DIR / 'exp3_weighted.h5'}")


✅ 콜백 설정 완료
  모델 저장 경로: ..\models\exp3_weighted.h5


## 8. 모델 학습 ⭐ (class_weight 적용)

실험 2와의 **유일한 코드 차이점**이 여기 있습니다.

`model.fit()`에 **`class_weight=class_weight_dict`** 파라미터를 추가합니다.

```python
# 실험 2 (class_weight 없음)
model.fit(train_generator, ...)

# 실험 3 (class_weight 적용) ⭐
model.fit(train_generator, ..., class_weight=class_weight_dict)
```

이 한 줄이 추가되면:
- 모델이 A1을 틀렸을 때 → A1 가중치만큼의 loss
- 모델이 A7을 틀렸을 때 → A7 가중치만큼의 loss
- **데이터가 적은 클래스를 틀리면 더 큰 벌점** → 소수 클래스 성능 향상


In [17]:
print("=" * 55)
print("🚀 실험 3: Transfer Learning + 증강 + Class Weight 학습 시작")
print("=" * 55)
print(f"\n적용된 class_weight:")
for idx, w in class_weight_dict.items():
    cls = lesion_classes[idx]
    print(f"  {cls}: {w:.3f}")
print()

# ⭐ 실험 3 핵심: class_weight 파라미터 추가
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weight_dict,    # ⭐ 이것만 추가됨!
    verbose=1
)

print("\n✅ 학습 완료!")
print(f"   실제 학습 에폭 수: {len(history.history['loss'])}회")


🚀 실험 3: Transfer Learning + 증강 + Class Weight 학습 시작

적용된 class_weight:
  A1: 1.000
  A2: 1.000
  A3: 1.000
  A4: 1.001
  A5: 1.000
  A6: 1.000
  A7: 1.000

Epoch 1/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1394 - loss: 1.9549
Epoch 1: val_loss improved from None to 1.94628, saving model to ..\models\exp3_weighted.h5



Epoch 1: finished saving model to ..\models\exp3_weighted.h5
383/383 ━━━━━━━━━━━━━━━━━━━━ 1485s 4s/step - accuracy: 0.1391 - loss: 1.9514 - val_accuracy: 0.1429 - val_loss: 1.9463 - learning_rate: 1.0000e-04
Epoch 2/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1389 - loss: 1.9478
Epoch 2: val_loss improved from 1.94628 to 1.94593, saving model to ..\models\exp3_weighted.h5



Epoch 2: finished saving model to ..\models\exp3_weighted.h5
383/383 ━━━━━━━━━━━━━━━━━━━━ 854s 2s/step - accuracy: 0.1394 - loss: 1.9471 - val_accuracy: 0.1429 - val_loss: 1.9459 - learning_rate: 1.0000e-04
Epoch 3/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1364 - loss: 1.9462
Epoch 3: val_loss did not improve from 1.94593
383/383 ━━━━━━━━━━━━━━━━━━━━ 700s 2s/step - accuracy: 0.1400 - loss: 1.9461 - val_accuracy: 0.1429 - val_loss: 1.9461 - learning_rate: 1.0000e-04
Epoch 4/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1464 - loss: 1.9460
Epoch 4: val_loss did not improve from 1.94593
383/383 ━━━━━━━━━━━━━━━━━━━━ 725s 2s/step - accuracy: 0.1445 - loss: 1.9460 - val_accuracy: 0.1429 - val_loss: 1.9474 - learning_rate: 1.0000e-04
Epoch 5/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1421 - loss: 1.9466
Epoch 5: val_loss did not improve from 1.94593

Epoch 5: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
383/383 ━━━━━━━━━━━━━━━━━━━


Epoch 6: finished saving model to ..\models\exp3_weighted.h5
383/383 ━━━━━━━━━━━━━━━━━━━━ 1026s 3s/step - accuracy: 0.1414 - loss: 1.9460 - val_accuracy: 0.1429 - val_loss: 1.9459 - learning_rate: 5.0000e-05
Epoch 7/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1390 - loss: 1.9459
Epoch 7: val_loss did not improve from 1.94591
383/383 ━━━━━━━━━━━━━━━━━━━━ 1087s 3s/step - accuracy: 0.1424 - loss: 1.9459 - val_accuracy: 0.1429 - val_loss: 1.9459 - learning_rate: 5.0000e-05
Epoch 8/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1450 - loss: 1.9459
Epoch 8: val_loss did not improve from 1.94591

Epoch 8: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.
383/383 ━━━━━━━━━━━━━━━━━━━━ 1010s 3s/step - accuracy: 0.1443 - loss: 1.9459 - val_accuracy: 0.1429 - val_loss: 1.9459 - learning_rate: 5.0000e-05
Epoch 9/20
218/383 ━━━━━━━━━━━━━━━━━━━━ 4:54 2s/step - accuracy: 0.1429 - loss: 1.9459

KeyboardInterrupt: 

## 9. 학습 곡선 시각화

실험 2와 비교하여 **Val Loss/Accuracy가 어떻게 달라졌는지** 확인합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Loss 곡선', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=12)
axes[0].grid(True)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_title('Accuracy 곡선', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=12)
axes[1].grid(True)

plt.suptitle('실험 3: Transfer Learning + 증강 + Class Weight 학습 곡선', fontsize=16, fontweight='bold')
plt.tight_layout()

plt.savefig(FIG_DIR / 'exp3_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 학습 곡선 저장 → {FIG_DIR / 'exp3_learning_curve.png'}")


## 10. 테스트 데이터 평가


In [ ]:
print("=" * 50)
print("📝 테스트 데이터 평가")
print("=" * 50)

test_loss, test_acc = model.evaluate(test_generator, verbose=1)

print(f"\n테스트 Loss    : {test_loss:.4f}")
print(f"테스트 Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")


## 11. 상세 평가 — Classification Report

실험 2 대비 **반려묘 관련 클래스의 Recall/F1이 올랐는지** 확인하는 것이 핵심입니다.


In [ ]:
y_pred_proba = model.predict(test_generator)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_generator.classes

class_names = list(train_generator.class_indices.keys())
LESION_MAP = {
    'A1': 'A1 구진/플라크',
    'A2': 'A2 비듬/각질',
    'A3': 'A3 태선화/색소',
    'A4': 'A4 농포/여드름',
    'A5': 'A5 미란/궤양',
    'A6': 'A6 결절/종괴',
    'A7': 'A7 무증상(정상)'
}
class_labels = [LESION_MAP[c] for c in class_names]

print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_labels))


### 11-1. Confusion Matrix (혼동 행렬)

실험 2 대비 **대각선 값이 더 고르게 분포**하는지 확인하세요.  
특정 클래스만 잘 맞추는 게 아니라, 모든 클래스를 공평하게 맞추는 것이 목표입니다.


In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel('예측 (Predicted)', fontsize=12)
ax.set_ylabel('실제 (Actual)', fontsize=12)
ax.set_title('실험 3: Transfer Learning + 증강 + Class Weight — Confusion Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig(FIG_DIR / 'exp3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 혼동 행렬 저장 → {FIG_DIR / 'exp3_confusion_matrix.png'}")


## 12. 종별(반려견/반려묘) 성능 분석 ⭐ (실험 3 전용)

Class Weight의 효과를 확인하기 위해, **반려견과 반려묘 각각의 정확도**를 측정합니다.  
실험 2 대비 **반려묘 정확도가 올랐는지**가 핵심 비교 포인트입니다.


In [ ]:
# 테스트 데이터에서 종별 인덱스 추출
test_species = df_test['species'].values

# 반려견(D)만 필터링
dog_mask = (test_species == 'D')
dog_acc = np.mean(y_pred[dog_mask] == y_true[dog_mask])

# 반려묘(C)만 필터링
cat_mask = (test_species == 'C')
cat_acc = np.mean(y_pred[cat_mask] == y_true[cat_mask])

print("=" * 50)
print("📊 종별 정확도 분석 (실험 3)")
print("=" * 50)
print(f"  반려견(D) 정확도: {dog_acc:.4f} ({dog_acc*100:.1f}%) — {dog_mask.sum():,}장")
print(f"  반려묘(C) 정확도: {cat_acc:.4f} ({cat_acc*100:.1f}%) — {cat_mask.sum():,}장")
print(f"  전체     정확도: {test_acc:.4f} ({test_acc*100:.1f}%)")
print()
print("💡 이 결과를 실험 2의 종별 정확도와 비교하여")
print("   Class Weight가 반려묘 성능 향상에 기여했는지 판단합니다.")


## 13. 실험 3 결과 요약


In [ ]:
print("=" * 60)
print("📋 실험 3: Transfer Learning + 증강 + Class Weight 결과 요약")
print("=" * 60)
print(f"모델 구조       : EfficientNetB0 (freeze) + Dense 분류층")
print(f"데이터 증강     : 적용 (회전, 반전, 밝기, 줌, 이동)")
print(f"Class Weight    : ✅ 적용 (질환별 가중치)")
print(f"학습률          : {LEARNING_RATE}")
print(f"학습 에폭       : {len(history.history['loss'])}회")
print(f"최종 Train Loss : {history.history['loss'][-1]:.4f}")
print(f"최종 Train Acc  : {history.history['accuracy'][-1]:.4f}")
print(f"최종 Val Loss   : {history.history['val_loss'][-1]:.4f}")
print(f"최종 Val Acc    : {history.history['val_accuracy'][-1]:.4f}")
print(f"테스트 Loss     : {test_loss:.4f}")
print(f"테스트 Accuracy : {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"반려견 정확도   : {dog_acc:.4f} ({dog_acc*100:.1f}%)")
print(f"반려묘 정확도   : {cat_acc:.4f} ({cat_acc*100:.1f}%)")
print(f"모델 저장 경로  : models/exp3_weighted.h5")
print("=" * 60)
print()
print("→ 다음: 실험 1, 2, 3 성능 비교 후 최종 모델 선정 → Streamlit 앱 탑재")
